# Extra weather features from the Open-Meteo archive (ERA5 reanalysis) - no login needed

Open-Meteo's free Historical Weather API serves ERA5 reanalysis data for any coordinates. Your records sit at only ~46 distinct locations, so ~46 requests give daily weather for all of them; the features are then computed locally.

**External source to cite on the Zindi forum:** Open-Meteo Historical Weather API (https://open-meteo.com/en/docs/historical-weather-api), based on ECMWF ERA5 reanalysis. The competition lists ERA5 as a permitted source.

Run the cells in order. Untested code: if a cell errors, paste the error to Claude.

In [1]:
# Cell 1 - install into this notebook's Python (restart kernel if it asks)
%pip install -q requests pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 2 - load records and group them into location cells
from pathlib import Path
import time, requests
import numpy as np, pandas as pd

ROOT = Path.cwd()
if not (ROOT / "Train.csv").exists():
    alt = ROOT / "Primary competition visual Climate Risk and Health Prediction Challenge"
    if (alt / "Train.csv").exists():
        ROOT = alt
print("Using folder:", ROOT)

tr = pd.read_csv(ROOT / "Train.csv"); te = pd.read_csv(ROOT / "Test.csv")
cols = ["ID", "latitude", "longitude", "deathdate"]
pts = pd.concat([tr[cols], te[cols]], ignore_index=True)
pts["date"] = pd.to_datetime(pts.deathdate).dt.normalize()
pts["lat_c"] = pts.latitude.round(2); pts["lon_c"] = pts.longitude.round(2)
cells = pts[["lat_c", "lon_c"]].drop_duplicates().reset_index(drop=True)
START = (pts.date.min() - pd.Timedelta(days=400)).strftime("%Y-%m-%d")   # 400 days of history for lags/climatology
END = pts.date.max().strftime("%Y-%m-%d")
print(len(cells), "location cells | fetching", START, "to", END)

Using folder: c:\Users\boogey\Documents\GitHub\Projects To The F\Primary competition visual Climate Risk and Health Prediction Challenge
46 location cells | fetching 2006-08-28 to 2022-12-13


In [4]:
    # Cell 3 - download daily weather per cell; cached cells are reused
URL = "https://archive-api.open-meteo.com/v1/archive"
DAILY = ["temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
             "apparent_temperature_max", "precipitation_sum", "rain_sum", "precipitation_hours",
             "wind_speed_10m_max", "shortwave_radiation_sum", "et0_fao_evapotranspiration"]
CACHE = ROOT / "openmeteo_cache"
CACHE.mkdir(exist_ok=True)

def fetch(lat, lon):
    f = CACHE / f"{lat}_{lon}.csv"
    if f.exists():
            return pd.read_csv(f, parse_dates=["date"])

    params = dict(
            latitude=lat, longitude=lon, start_date=START, end_date=END,
            daily=",".join(DAILY), timezone="GMT"
        )

    for attempt in range(6):
            try:
                r = requests.get(URL, params=params, timeout=(20, 300))

                if r.status_code == 200:
                    data = r.json()
                    if "daily" not in data:
                        raise RuntimeError(f"Unexpected API response: {data}")
                    d = pd.DataFrame(data["daily"]).rename(columns={"time": "date"})
                    d["date"] = pd.to_datetime(d["date"])
                    d.to_csv(f, index=False)
                    time.sleep(1)
                    return d

                if r.status_code not in (429, 500, 502, 503, 504):
                    raise RuntimeError(
                        f"HTTP {r.status_code} for {lat},{lon}: {r.text[:300]}"
                    )

            except requests.exceptions.RequestException as e:
                if attempt == 5:
                    raise RuntimeError(f"Request failed for {lat},{lon} after retries") from e

            if attempt == 5:
                raise RuntimeError(f"API remained unavailable for {lat},{lon} after retries")

            wait = min(30 * (2 ** attempt), 180)
            print(f"Retrying {lat},{lon} in {wait}s (attempt {attempt + 2}/6)")
            time.sleep(wait)

    daily = {}
    for i, row in cells.iterrows():
        daily[(row.lat_c, row.lon_c)] = fetch(row.lat_c, row.lon_c)
        if (i + 1) % 10 == 0:
            print(f"{i + 1}/{len(cells)} cells done")

    print("all downloaded")
    daily[(cells.lat_c[0], cells.lon_c[0])].head()
    if (i + 1) % 10 == 0: print(f"{i + 1}/{len(cells)} cells done")
print("all downloaded")
daily[(cells.lat_c[0], cells.lon_c[0])].head()

all downloaded


,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,apparent_temperature_max,precipitation_sum,rain_sum,precipitation_hours,wind_speed_10m_max,shortwave_radiation_sum,et0_fao_evapotranspiration
0,2006-08-28,27.5,16.9,22.2,31.5,3.9,3.9,4.0,13.7,22.91,4.62
1,2006-08-29,25.0,17.8,21.1,29.3,4.9,4.9,10.0,7.1,18.77,3.60
2,2006-08-30,26.8,17.4,22.0,31.7,1.6,1.6,7.0,8.8,21.94,4.21
3,2006-08-31,26.5,18.5,22.0,30.4,4.5,4.5,6.0,9.2,20.31,3.94
4,2006-09-01,27.6,17.7,22.6,30.3,0.7,0.7,3.0,14.6,23.51,4.77


In [5]:
# Cell 4 - build rolling-window features per cell (all computed locally)
WINDOWS = [3, 7, 14, 30, 60, 90, 180]

def build(d):
    d = d.sort_values("date").reset_index(drop=True)
    rain = d.precipitation_sum.fillna(0); n = len(d)
    out = pd.DataFrame({"date": d.date})
    for w in WINDOWS:
        out[f"om_rain_sum_{w}d"] = rain.rolling(w, min_periods=w).sum()
        out[f"om_rain_max_{w}d"] = rain.rolling(w, min_periods=w).max()
        out[f"om_rain_days_{w}d"] = (rain > 1).astype(float).rolling(w, min_periods=w).sum()
        out[f"om_tmean_{w}d"] = d.temperature_2m_mean.rolling(w, min_periods=w).mean()
        out[f"om_tmax_{w}d"] = d.temperature_2m_max.rolling(w, min_periods=w).max()
        out[f"om_tmin_{w}d"] = d.temperature_2m_min.rolling(w, min_periods=w).min()
        out[f"om_hot_days_{w}d"] = (d.temperature_2m_max >= 30).astype(float).rolling(w, min_periods=w).sum()
    for w in [7, 30]:
        out[f"om_solar_{w}d"] = d.shortwave_radiation_sum.rolling(w, min_periods=w).mean()
        out[f"om_et0_{w}d"] = d.et0_fao_evapotranspiration.rolling(w, min_periods=w).sum()
        out[f"om_wind_{w}d"] = d.wind_speed_10m_max.rolling(w, min_periods=w).mean()
        out[f"om_apptmax_{w}d"] = d.apparent_temperature_max.rolling(w, min_periods=w).mean()
    last_rain = pd.Series(np.where(rain > 1, np.arange(n), np.nan)).ffill()
    out["om_dry_spell"] = (np.arange(n) - last_rain).fillna(365).clip(upper=365)
    # anomalies vs this location's own same-day-of-year average (over the available years)
    doy = d.date.dt.dayofyear
    for col in ["om_rain_sum_30d", "om_rain_sum_90d", "om_tmean_30d", "om_tmean_90d"]:
        out[col.replace("om_", "om_anom_")] = out[col] - out.groupby(doy)[col].transform("mean")
    return out

frames = []
for (lat, lon), d in daily.items():
    f = build(d); f["lat_c"], f["lon_c"] = lat, lon; frames.append(f)
allf = pd.concat(frames, ignore_index=True)
extra = pts[["ID", "date", "lat_c", "lon_c"]].merge(allf, on=["date", "lat_c", "lon_c"], how="left")
extra = extra.drop(columns=["date", "lat_c", "lon_c"])
print(extra.shape, "| rows missing all features:", extra.drop(columns="ID").isna().all(axis=1).sum())
extra.head()

(4176, 63) | rows missing all features: 2294


,ID,om_rain_sum_3d,om_rain_max_3d,om_rain_days_3d,om_tmean_3d,om_tmax_3d,om_tmin_3d,om_hot_days_3d,om_rain_sum_7d,om_rain_max_7d,...,om_apptmax_7d,om_solar_30d,om_et0_30d,om_wind_30d,om_apptmax_30d,om_dry_spell,om_anom_rain_sum_30d,om_anom_rain_sum_90d,om_anom_tmean_30d,om_anom_tmean_90d
0,ID_1FE951BD,10.5,6.2,3.0,21.566667,27.5,17.5,0.0,16.2,6.2,...,30.328571,20.630000,119.70,11.486667,30.253333,0.0,-19.988235,41.71875,-0.345490,-0.382361
1,ID_5927A443,0.3,0.3,0.0,23.300000,29.3,17.7,0.0,1.1,0.7,...,31.471429,21.438667,127.70,10.453333,29.760000,10.0,-41.743750,-136.59375,0.141875,0.166736
2,ID_C4E67025,13.7,8.8,2.0,21.800000,27.5,18.0,0.0,49.6,19.7,...,29.814286,19.822333,117.61,11.663333,29.623333,1.0,73.412500,24.04375,-1.017500,-0.167500
3,ID_44DC5CE9,6.3,4.9,2.0,22.100000,28.1,17.8,0.0,38.8,19.7,...,30.571429,20.056000,118.99,11.680000,29.573333,0.0,61.937500,-21.38125,-0.878333,-0.105069
4,ID_33C677BB,36.2,21.8,3.0,20.233333,26.2,12.9,0.0,142.9,40.8,...,26.685714,19.534667,116.36,9.020000,26.303333,0.0,277.406250,205.79375,-1.998333,-0.892917


In [6]:
# Cell 5 - save, then upload climate_openmeteo.csv to Claude
extra.to_csv(ROOT / "climate_openmeteo.csv", index=False)
print("saved", ROOT / "climate_openmeteo.csv")

saved c:\Users\boogey\Documents\GitHub\Projects To The F\Primary competition visual Climate Risk and Health Prediction Challenge\climate_openmeteo.csv
